# Воркшоп №5 — Вычислительная сложность

**Курсовая:** «Использование мультимодальных эмбеддингов в рекомендательных системах».

Замеряю:
1. Размер артефактов на диске
2. Время инференса на один трек (CPU)
3. RAM для матриц эмбеддингов

Результат — `data/results/compute_report.csv`.

In [1]:
from __future__ import annotations

import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

DEVICE = "cpu"
SAMPLE_N = 20          # сколько треков замерять для усреднения (не греем Mac)
BATCH_SIZE = 4

DATA_DIR = Path("data")
EMB_DIR = DATA_DIR / "embeddings"
FUSION_DIR = DATA_DIR / "fusion"
RESULTS_DIR = DATA_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)

Device: cpu


## 1. Размер артефактов на диске

In [2]:
def dir_size(path: Path) -> int:
    if path.is_file():
        return path.stat().st_size
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file())


def human_mb(n: int) -> float:
    return round(n / 1024 / 1024, 2)


artifacts = [
    ("metadata.parquet", DATA_DIR / "metadata.parquet"),
    ("edges.parquet", DATA_DIR / "edges.parquet"),
    ("audio/", DATA_DIR / "audio"),
    ("images/", DATA_DIR / "images"),
    ("embeddings/text.npz", EMB_DIR / "text.npz"),
    ("embeddings/image.npz", EMB_DIR / "image.npz"),
    ("embeddings/audio.npz", EMB_DIR / "audio.npz"),
    ("fusion/early_fusion.npz", FUSION_DIR / "early_fusion.npz"),
    ("fusion/cross_attention.npz", FUSION_DIR / "cross_attention.npz"),
    ("fusion/cross_attention.pt", FUSION_DIR / "cross_attention.pt"),
    ("results/eval_results.csv", RESULTS_DIR / "eval_results.csv"),
]

size_rows = []
for name, p in artifacts:
    if p.exists():
        b = dir_size(p)
        size_rows.append({"artifact": name, "size_mb": human_mb(b)})
    else:
        size_rows.append({"artifact": name, "size_mb": None})

size_df = pd.DataFrame(size_rows)
print(size_df.to_string(index=False))
print(f"\nИТОГО data/: {human_mb(dir_size(DATA_DIR))} МБ")

                  artifact  size_mb
          metadata.parquet     8.95
             edges.parquet    21.87
                    audio/ 14855.19
                   images/  1520.35
       embeddings/text.npz    40.21
      embeddings/image.npz    21.85
      embeddings/audio.npz    26.83
   fusion/early_fusion.npz    91.26
fusion/cross_attention.npz    13.43
 fusion/cross_attention.pt     3.77
  results/eval_results.csv     0.00



ИТОГО data/: 17140.38 МБ


## 2. Время инференса

In [3]:
import subprocess
import imageio_ffmpeg
from PIL import Image
from transformers import AutoTokenizer, AutoModel, CLIPModel, CLIPProcessor, ClapModel, ClapProcessor

meta = pd.read_parquet(DATA_DIR / "metadata.parquet")
mask = meta["has_audio"].fillna(False) & meta["has_image"].fillna(False)
meta_ok = meta.loc[mask].head(SAMPLE_N)
N_total = int(mask.sum())

FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()

def load_audio(path):
    cmd = [FFMPEG, "-v", "quiet", "-i", str(path), "-t", "30", "-f", "f32le", "-ac", "1", "-ar", "48000", "-"]
    raw = subprocess.run(cmd, check=True, stdout=subprocess.PIPE).stdout
    return np.frombuffer(raw, dtype=np.float32).copy()

tok = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-base")
text_model = AutoModel.from_pretrained("intfloat/multilingual-e5-base").to(DEVICE).eval()
text_times = []
with torch.no_grad():
    for _, row in meta_ok.iterrows():
        t0 = time.perf_counter()
        enc = tok([f"passage: {row['text_blob']}"], padding=True, truncation=True, max_length=512, return_tensors="pt")
        out = text_model(**{k: v.to(DEVICE) for k, v in enc.items()})
        _ = out.last_hidden_state.mean(dim=1)
        text_times.append(time.perf_counter() - t0)
text_ms = np.mean(text_times) * 1000

clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE).eval()
image_times = []
with torch.no_grad():
    for _, row in meta_ok.iterrows():
        t0 = time.perf_counter()
        img = Image.open(row["image_path"]).convert("RGB")
        enc = clip_proc(images=[img], return_tensors="pt")
        _ = clip_model.get_image_features(**{k: v.to(DEVICE) for k, v in enc.items()})
        image_times.append(time.perf_counter() - t0)
image_ms = np.mean(image_times) * 1000

clap_proc = ClapProcessor.from_pretrained("laion/clap-htsat-unfused")
clap_model = ClapModel.from_pretrained("laion/clap-htsat-unfused").to(DEVICE).eval()
audio_times = []
with torch.no_grad():
    for _, row in meta_ok.iterrows():
        t0 = time.perf_counter()
        wav = load_audio(row["audio_path"])
        enc = clap_proc(audio=[wav], sampling_rate=48000, return_tensors="pt")
        _ = clap_model.get_audio_features(**{k: v.to(DEVICE) for k, v in enc.items()})
        audio_times.append(time.perf_counter() - t0)
audio_ms = np.mean(audio_times) * 1000

compute_rows = [
    {"stage": "text_e5", "ms_per_track": round(text_ms, 1), "est_total_min": round(text_ms * N_total / 1000 / 60, 1)},
    {"stage": "image_clip", "ms_per_track": round(image_ms, 1), "est_total_min": round(image_ms * N_total / 1000 / 60, 1)},
    {"stage": "audio_clap", "ms_per_track": round(audio_ms, 1), "est_total_min": round(audio_ms * N_total / 1000 / 60, 1)},
]
compute_df = pd.DataFrame(compute_rows)
compute_df["n_tracks"] = N_total
print(compute_df.to_string(index=False))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/447 [00:00<?, ?it/s]

     stage  ms_per_track  est_total_min  n_tracks
   text_e5         109.9           27.1     14787
image_clip          94.1           23.2     14787
audio_clap         139.8           34.5     14787


## 3. Память эмбеддингов в RAM

In [4]:
def emb_ram_mb(shape, dtype=np.float32):
    return shape[0] * shape[1] * np.dtype(dtype).itemsize / 1024 / 1024

text = np.load(EMB_DIR / "text.npz")["emb"]
image = np.load(EMB_DIR / "image.npz")["emb"]
audio = np.load(EMB_DIR / "audio.npz")["emb"]
early = np.load(FUSION_DIR / "early_fusion.npz")["emb"]
cross = np.load(FUSION_DIR / "cross_attention.npz")["emb"]

ram_rows = [
    {"matrix": "text", "shape": str(text.shape), "ram_mb": round(emb_ram_mb(text.shape), 1)},
    {"matrix": "image", "shape": str(image.shape), "ram_mb": round(emb_ram_mb(image.shape), 1)},
    {"matrix": "audio", "shape": str(audio.shape), "ram_mb": round(emb_ram_mb(audio.shape), 1)},
    {"matrix": "early_fusion", "shape": str(early.shape), "ram_mb": round(emb_ram_mb(early.shape), 1)},
    {"matrix": "cross_attention", "shape": str(cross.shape), "ram_mb": round(emb_ram_mb(cross.shape), 1)},
]
ram_df = pd.DataFrame(ram_rows)
print(ram_df.to_string(index=False))
print(f"\nСуммарно эмбеддинги в RAM: {ram_df['ram_mb'].sum():.1f} МБ")

         matrix         shape  ram_mb
           text  (14787, 768)    43.3
          image  (14787, 512)    28.9
          audio  (14787, 512)    28.9
   early_fusion (14787, 1792)   101.1
cross_attention  (14787, 256)    14.4

Суммарно эмбеддинги в RAM: 216.6 МБ


In [5]:
OUT = RESULTS_DIR / "compute_report.csv"
summary = pd.concat([
    size_df.assign(section="disk_size"),
    compute_df.assign(section="inference_time"),
    ram_df.assign(section="ram_usage"),
], ignore_index=True)
summary.to_csv(OUT, index=False)
print(f"Saved: {OUT}")

Saved: data/results/compute_report.csv
